# 面试问题：怎样从零实现 Vision Transformer，并解释 Patch、MHSA 与位置编码？

## 可直接复述的回答主线

1. ViT 把图像切成不重叠 patch，将每块展平后用共享线性层映射成 token。
2. 序列前加入可学习 CLS token，并为每个 patch 序号加入位置向量，否则模型只看到无序 patch 集合。
3. 多头自注意力手写为 QKᵀ/√d、softmax 和 AV，不同 head 在不同特征子空间建立全局关系。
4. EncoderBlock 使用 LayerNorm、attention、MLP 和两条残差，MLP 只逐 token 变换，token 间混合发生在 attention。
5. 评测应展示 patch 内容、token shape、attention 行和与 CLS 注意力网格，并与同数据全局统计基线比较。
6. 移除位置编码后，具有相同 patch 多重集合但位置不同的图像会得到相同 CLS 输出，这是可复现的结构失败。
7. 生产还需大规模预训练、增强、位置插值、长 token 显存、FlashAttention、蒸馏、量化和域漂移评测。

下面用同一批可读输入依次验证朴素基线、手写核心机制、中间过程、失败修正与生产边界。

## 1. 真实案例与输入预览

案例是 12 张 16×16 脱敏文档小图，亮色 4×4 印章分别位于左上、右上或底部中间，每类 4 个亮度版本。相同版本的三类图拥有完全相同的 patch 内容多重集合，仅位置不同，因此可直接验证位置编码是否真的生效。

In [1]:
import math  # 计算 attention 缩放与训练梯度。
import warnings  # 过滤本地 PyTorch 环境的无关兼容警告。
warnings.filterwarnings("ignore", message=".*pynvml.*", category=FutureWarning)  # 保持输出聚焦 ViT 机制。
import torch  # 使用基础张量和线性层手写 Vision Transformer。
torch.manual_seed(241)  # 固定模型初始化和训练轨迹。
torch.set_num_threads(1)  # 固定 CPU 单线程提高可复现性。
class_names = ["左上印章", "右上印章", "底部印章"]  # 定义三个只由空间位置区分的类别。
patch_positions = [(0, 0), (0, 3), (3, 1)]  # 定义三类在四乘四 patch 网格中的位置。
images = []  # 保存十二张单通道文档图。
labels = []  # 保存三类位置标签。
sample_ids = []  # 保存可读图像编号。
for class_index, (patch_row, patch_column) in enumerate(patch_positions):  # 依次生成三个印章位置类别。
    for variant in range(4):  # 为每类生成四个相同布局的亮度版本。
        background_level = 0.03 + 0.01 * variant  # 设置全图一致的弱纸张底色。
        stamp_level = 0.65 + 0.08 * variant  # 设置印章像素亮度。
        image = torch.full((16, 16), background_level)  # 创建均匀文档背景以保证 patch 集合可置换。
        top = patch_row * 4  # 把 patch 行号转换为像素起点。
        left = patch_column * 4  # 把 patch 列号转换为像素起点。
        image[top:top + 4, left:left + 4] = stamp_level  # 写入一个完整四乘四亮色 patch。
        images.append(image.unsqueeze(0))  # 保存单通道文档图。
        labels.append(class_index)  # 保存印章位置类别。
        sample_ids.append(f"doc-{class_index}-{variant}")  # 生成稳定样本 ID。
images = torch.stack(images)  # 堆叠为十二乘一乘十六乘十六张量。
labels = torch.tensor(labels, dtype=torch.long)  # 转换为分类标签张量。
def render_image(image):  # 把文档小图转为字符画。
    return "\n".join("".join("#" if float(pixel) > 0.5 else "." for pixel in row) for row in image.squeeze(0))  # 用井号展示印章像素。
print("教学实验输入：12张印章位置图，shape=", tuple(images.shape))  # 展示批量、通道和空间尺寸。
for index in range(len(images)):  # 逐图展示真实位置模式和亮度版本。
    print(f"\n{sample_ids[index]} label={class_names[labels[index]]} mean={images[index].mean().item():.4f}\n{render_image(images[index])}")  # 输出当前文档字符图与字段。

教学实验输入：12张印章位置图，shape= (12, 1, 16, 16)

doc-0-0 label=左上印章 mean=0.0687
####............
####............
####............
####............
................
................
................
................
................
................
................
................
................
................
................
................

doc-0-1 label=左上印章 mean=0.0831
####............
####............
####............
####............
................
................
................
................
................
................
................
................
................
................
................
................

doc-0-2 label=左上印章 mean=0.0975
####............
####............
####............
####............
................
................
................
................
................
................
................
................
................
................
................
................

doc-0-3 label=左上印章 mean=0.1119
####............


## 2. Baseline / 基线：只用全图平均亮度

每个类别都包含相同四种背景/印章亮度，因此类平均亮度完全一致。最近亮度中心无法知道印章在哪个 patch，并列时固定预测第一类。

In [2]:
global_means = images.mean(dim=(1, 2, 3))  # 提取每张文档唯一的全局亮度特征。
class_brightness = torch.stack([global_means[labels == class_index].mean() for class_index in range(3)])  # 计算每类平均亮度中心。
baseline_distances = torch.abs(global_means[:, None] - class_brightness[None, :])  # 计算样本到三个亮度中心的距离。
baseline_predictions = baseline_distances.argmin(dim=1)  # 对并列最小距离固定选择首类别。
baseline_accuracy = float((baseline_predictions == labels).to(torch.float32).mean().item())  # 计算同一十二张图的基线准确率。
print("Baseline：global mean brightness")  # 标记下表为空间信息缺失方案。
print("sample   true       mean      class_centers                 prediction")  # 输出逐样本基线表头。
for index in range(len(images)):  # 逐图展示亮度无法定位的结果。
    print(f"{sample_ids[index]:<8} {class_names[labels[index]]:<8} {global_means[index].item():.5f} {class_brightness.tolist()} {class_names[baseline_predictions[index]]}")  # 输出真实类、亮度中心和基线预测。
print(f"Baseline accuracy={baseline_accuracy:.4f}")  # 展示只看全局统计的类别先验水平。

Baseline：global mean brightness
sample   true       mean      class_centers                 prediction
doc-0-0  左上印章     0.06875 [0.09031249582767487, 0.09031249582767487, 0.09031249582767487] 左上印章
doc-0-1  左上印章     0.08313 [0.09031249582767487, 0.09031249582767487, 0.09031249582767487] 左上印章
doc-0-2  左上印章     0.09750 [0.09031249582767487, 0.09031249582767487, 0.09031249582767487] 左上印章
doc-0-3  左上印章     0.11187 [0.09031249582767487, 0.09031249582767487, 0.09031249582767487] 左上印章
doc-1-0  右上印章     0.06875 [0.09031249582767487, 0.09031249582767487, 0.09031249582767487] 左上印章
doc-1-1  右上印章     0.08312 [0.09031249582767487, 0.09031249582767487, 0.09031249582767487] 左上印章
doc-1-2  右上印章     0.09750 [0.09031249582767487, 0.09031249582767487, 0.09031249582767487] 左上印章
doc-1-3  右上印章     0.11187 [0.09031249582767487, 0.09031249582767487, 0.09031249582767487] 左上印章
doc-2-0  底部印章     0.06875 [0.09031249582767487, 0.09031249582767487, 0.09031249582767487] 左上印章
doc-2-1  底部印章     0.08312 [0.0903124958276

## 3. 底层实现：手工 Patchify、多头自注意力、EncoderBlock 与完整 ViT

不使用 `nn.MultiheadAttention`、`nn.Transformer`、timm 或 torchvision。Patch 用 `unfold` 明确切分；Q/K/V、head reshape、scaled dot-product、softmax、残差和 MLP 全部写在 `forward` 中。

In [3]:
class PatchEmbedding(torch.nn.Module):  # 定义显式切块与共享线性投影。
    def __init__(self, image_size=16, patch_size=4, embedding_dim=24):  # 初始化 patch 网格和投影参数。
        super().__init__()  # 注册线性投影参数。
        self.image_size = image_size  # 保存固定输入边长。
        self.patch_size = patch_size  # 保存不重叠 patch 边长。
        self.grid_size = image_size // patch_size  # 计算每个方向 patch 数。
        self.patch_count = self.grid_size ** 2  # 计算总 patch token 数。
        self.projection = torch.nn.Linear(patch_size * patch_size, embedding_dim)  # 把十六像素 patch 映射到 token 维度。
    def forward(self, inputs, return_patches=False):  # 切分 NCHW 图像并按需返回原始 patch。
        patches = inputs.unfold(2, self.patch_size, self.patch_size).unfold(3, self.patch_size, self.patch_size)  # 生成批次乘通道乘网格高乘网格宽乘 patch 高宽视图。
        patches = patches.permute(0, 2, 3, 1, 4, 5).contiguous()  # 把网格位置移到通道和像素之前。
        flat_patches = patches.view(inputs.shape[0], self.patch_count, -1)  # 展平每个四乘四单通道 patch。
        tokens = self.projection(flat_patches)  # 对每个位置共享同一线性投影。
        return (tokens, flat_patches) if return_patches else tokens  # 按需返回像素级 patch 证据。
class MultiHeadSelfAttention(torch.nn.Module):  # 定义不调用现成 attention 的多头自注意力。
    def __init__(self, embedding_dim=24, head_count=4):  # 初始化 QKV 与输出投影。
        super().__init__()  # 注册注意力参数。
        self.embedding_dim = embedding_dim  # 保存 token 总维度。
        self.head_count = head_count  # 保存并行 head 数。
        self.head_dim = embedding_dim // head_count  # 计算每个 head 子空间维度。
        self.qkv = torch.nn.Linear(embedding_dim, 3 * embedding_dim)  # 一次生成 query、key 和 value。
        self.output = torch.nn.Linear(embedding_dim, embedding_dim)  # 合并多头后重新混合通道。
    def forward(self, tokens, return_attention=False):  # 手写 scaled dot-product attention 前向。
        batch_size, token_count, _ = tokens.shape  # 读取批量和 token 数。
        qkv = self.qkv(tokens).view(batch_size, token_count, 3, self.head_count, self.head_dim)  # 切分 QKV 与多头子空间。
        qkv = qkv.permute(2, 0, 3, 1, 4)  # 调整为三乘批次乘 head 乘 token 乘 head_dim。
        queries, keys, values = qkv.unbind(dim=0)  # 解包三类注意力向量。
        scores = queries @ keys.transpose(-2, -1) / math.sqrt(self.head_dim)  # 计算缩放后的 token 两两相似度。
        attention = torch.softmax(scores, dim=-1)  # 沿被关注 token 维归一化概率。
        context = attention @ values  # 按注意力权重聚合 value。
        context = context.transpose(1, 2).contiguous().view(batch_size, token_count, self.embedding_dim)  # 合并多个 head 回 token 表示。
        output = self.output(context)  # 执行最终通道投影。
        return (output, attention) if return_attention else output  # 按需返回四维注意力矩阵。
class EncoderBlock(torch.nn.Module):  # 定义 pre-norm ViT 编码块。
    def __init__(self, embedding_dim=24, head_count=4, mlp_dim=48):  # 初始化两条归一化残差子层。
        super().__init__()  # 注册 attention 和 MLP 参数。
        self.norm_one = torch.nn.LayerNorm(embedding_dim)  # 为 attention 子层执行 pre-norm。
        self.attention = MultiHeadSelfAttention(embedding_dim, head_count)  # 创建手写多头注意力。
        self.norm_two = torch.nn.LayerNorm(embedding_dim)  # 为 MLP 子层执行 pre-norm。
        self.mlp = torch.nn.Sequential(torch.nn.Linear(embedding_dim, mlp_dim), torch.nn.GELU(), torch.nn.Linear(mlp_dim, embedding_dim))  # 对每个 token 独立升维再降维。
    def forward(self, tokens, return_attention=False):  # 执行两条残差并按需返回注意力。
        attention_output, attention = self.attention(self.norm_one(tokens), return_attention=True)  # 计算归一化 token 的全局交互。
        tokens = tokens + attention_output  # 添加第一条 attention 残差。
        tokens = tokens + self.mlp(self.norm_two(tokens))  # 添加第二条逐 token MLP 残差。
        return (tokens, attention) if return_attention else tokens  # 按需返回 attention 权重。
class VisionTransformer(torch.nn.Module):  # 定义 patch、CLS、位置、编码器和分类头完整计算图。
    def __init__(self, class_count=3, embedding_dim=24, depth=2):  # 初始化小型教学 ViT。
        super().__init__()  # 注册全部网络参数。
        self.patch_embedding = PatchEmbedding(16, 4, embedding_dim)  # 创建十六 patch 的共享投影。
        self.class_token = torch.nn.Parameter(torch.zeros(1, 1, embedding_dim))  # 创建可学习 CLS 汇聚 token。
        self.position_embedding = torch.nn.Parameter(torch.zeros(1, 17, embedding_dim))  # 为 CLS 加十六 patch 保存位置向量。
        self.blocks = torch.nn.ModuleList([EncoderBlock(embedding_dim, 4, 48) for _ in range(depth)])  # 创建两层手写 Transformer block。
        self.norm = torch.nn.LayerNorm(embedding_dim)  # 归一化最终 CLS 表示。
        self.head = torch.nn.Linear(embedding_dim, class_count)  # 把 CLS 映射到三类 logits。
        torch.nn.init.normal_(self.position_embedding, std=0.02)  # 用小随机值初始化位置差异。
        torch.nn.init.normal_(self.class_token, std=0.02)  # 初始化 CLS 内容向量。
    def forward(self, inputs, use_position=True, return_debug=False):  # 执行完整 ViT 并允许失败实验关闭位置编码。
        patch_tokens, flat_patches = self.patch_embedding(inputs, return_patches=True)  # 取得十六个 patch token 和原像素。
        class_tokens = self.class_token.expand(inputs.shape[0], -1, -1)  # 为批次复制共享 CLS token。
        tokens = torch.cat([class_tokens, patch_tokens], dim=1)  # 把 CLS 放在序列首位。
        tokens = tokens + self.position_embedding if use_position else tokens  # 根据实验开关加入十七个位置向量。
        attention_maps = []  # 保存每层多头注意力供解释。
        for block in self.blocks:  # 顺序执行两个 EncoderBlock。
            tokens, attention = block(tokens, return_attention=True)  # 更新 token 并读取当前注意力矩阵。
            attention_maps.append(attention)  # 保存当前层注意力证据。
        normalized = self.norm(tokens)  # 归一化全部最终 token。
        logits = self.head(normalized[:, 0])  # 只读取 CLS token 执行分类。
        debug = {"flat_patches": flat_patches, "tokens": tokens, "normalized": normalized, "attention_maps": attention_maps}  # 汇总 patch、token 和注意力中间量。
        return (logits, debug) if return_debug else logits  # 按需返回完整解释数据。
model = VisionTransformer()  # 创建待训练的手写 ViT。
optimizer = torch.optim.Adam(model.parameters(), lr=0.012)  # 创建 Transformer 参数优化器。
history = []  # 保存真实 backward 的损失和梯度轨迹。
for step in range(320):  # 在十二张位置图上执行全批次训练。
    optimizer.zero_grad(set_to_none=True)  # 清除上一步全部参数梯度。
    logits, training_debug = model(images, return_debug=True)  # 前向执行 patch、attention、MLP 和分类头。
    loss = torch.nn.functional.cross_entropy(logits, labels)  # 计算三类位置交叉熵。
    loss.backward()  # 对 patch 投影、位置、attention 和 MLP 真实反向传播。
    gradient_norm = math.sqrt(sum(float((parameter.grad ** 2).sum().item()) for parameter in model.parameters() if parameter.grad is not None))  # 汇总全部非空梯度二范数。
    optimizer.step()  # 应用 Adam 更新 ViT 参数。
    if step % 80 == 0 or step == 319:  # 每八十步保存训练证据。
        accuracy = float((logits.argmax(dim=1) == labels).to(torch.float32).mean().item())  # 计算当前批次分类准确率。
        history.append({"step": step, "loss": loss.item(), "accuracy": accuracy, "gradient_norm": gradient_norm})  # 保存损失、准确率和梯度。
model.eval()  # 切换到确定性推理模式。
with torch.no_grad():  # 取得最终 logits、patch 和注意力矩阵。
    final_logits, final_debug = model(images, return_debug=True)  # 对十二张文档执行完整 ViT 推理。
final_attention = final_debug["attention_maps"][-1]  # 读取最后 EncoderBlock 的多头注意力。
class_attention_grid = final_attention[0, 0, 0, 1:].view(4, 4)  # 把首图 head0 的 CLS 到 patch 权重恢复为空间网格。
print("ViT训练轨迹=", history)  # 展示 loss、准确率和非零梯度。
print("doc-0-0 flat patch sums=", final_debug["flat_patches"][0].sum(dim=1).tolist())  # 展示哪个 patch 真正包含印章像素。
print("token/attention shapes=", tuple(final_debug["tokens"].shape), tuple(final_attention.shape))  # 展示十七 token 和四头注意力张量。
print("最后层head0 CLS→patch 4x4网格=", torch.round(class_attention_grid * 10000) / 10000)  # 展示 CLS 对空间位置的注意力分布。
print("attention首行和=", final_attention[0, :, 0].sum(dim=-1).tolist())  # 验证每个 head 的 CLS softmax 概率和为一。

ViT训练轨迹= [{'step': 0, 'loss': 1.1775623559951782, 'accuracy': 0.3333333432674408, 'gradient_norm': 2.052462070671617}, {'step': 80, 'loss': 0.0014366301475092769, 'accuracy': 1.0, 'gradient_norm': 0.009698561900540097}, {'step': 160, 'loss': 0.00021916611876804382, 'accuracy': 1.0, 'gradient_norm': 0.001408685842880275}, {'step': 240, 'loss': 0.0001267298503080383, 'accuracy': 1.0, 'gradient_norm': 0.0008207795355649611}, {'step': 319, 'loss': 8.50616124807857e-05, 'accuracy': 1.0, 'gradient_norm': 0.000553402410636274}]
doc-0-0 flat patch sums= [10.40000057220459, 0.47999998927116394, 0.47999998927116394, 0.47999998927116394, 0.47999998927116394, 0.47999998927116394, 0.47999998927116394, 0.47999998927116394, 0.47999998927116394, 0.47999998927116394, 0.47999998927116394, 0.47999998927116394, 0.47999998927116394, 0.47999998927116394, 0.47999998927116394, 0.47999998927116394]
token/attention shapes= (12, 17, 24) (12, 4, 17, 17)
最后层head0 CLS→patch 4x4网格= tensor([[0.3008, 0.0044, 0.0060, 0

## 4. 逐图分类结果与结果解读

比较相同十二张图的全局亮度基线与 ViT，输出每张图的印章位置预测和最大概率。注意这里只证明核心计算图能学习受控位置任务。

In [4]:
final_probabilities = torch.softmax(final_logits, dim=1)  # 把最终 logits 转为可读三类概率。
final_predictions = final_probabilities.argmax(dim=1)  # 取得每张图最高概率类别。
vit_accuracy = float((final_predictions == labels).to(torch.float32).mean().item())  # 计算 ViT 同数据准确率。
print("sample   true       brightness_baseline  ViT_prediction  confidence")  # 输出逐图同数据结果表头。
for index in range(len(images)):  # 逐图展示位置分类结果。
    confidence = float(final_probabilities[index, final_predictions[index]].item())  # 读取当前最大类别概率。
    print(f"{sample_ids[index]:<8} {class_names[labels[index]]:<8} {class_names[baseline_predictions[index]]:<19} {class_names[final_predictions[index]]:<14} {confidence:.4f}")  # 输出真实位置、两种预测和置信度。
print(f"结果解读：global-mean baseline accuracy={baseline_accuracy:.4f}，手写ViT={vit_accuracy:.4f}；位置向量让相同patch集合可以按序号区分。")  # 解释位置编码带来的同数据收益。

sample   true       brightness_baseline  ViT_prediction  confidence
doc-0-0  左上印章     左上印章                左上印章           0.9999
doc-0-1  左上印章     左上印章                左上印章           0.9999
doc-0-2  左上印章     左上印章                左上印章           0.9999
doc-0-3  左上印章     左上印章                左上印章           0.9999
doc-1-0  右上印章     左上印章                右上印章           0.9999
doc-1-1  右上印章     左上印章                右上印章           1.0000
doc-1-2  右上印章     左上印章                右上印章           1.0000
doc-1-3  右上印章     左上印章                右上印章           1.0000
doc-2-0  底部印章     左上印章                底部印章           0.9999
doc-2-1  底部印章     左上印章                底部印章           0.9999
doc-2-2  底部印章     左上印章                底部印章           0.9999
doc-2-3  底部印章     左上印章                底部印章           0.9999
结果解读：global-mean baseline accuracy=0.3333，手写ViT=1.0000；位置向量让相同patch集合可以按序号区分。


## 5. 失败案例与修正：移除位置编码后交换 patch 不改变 CLS

`doc-0-0` 与 `doc-1-0` 的像素 patch 多重集合完全相同，只是亮 patch 从左上换到右上。关闭位置向量时 Transformer 对 patch 置换等变，CLS 分类输出相同；恢复位置向量后 logits 明显不同。

In [5]:
comparison_images = images[[0, 4]]  # 选择相同亮度但印章 patch 位置不同的两张图。
with torch.no_grad():  # 在无梯度环境对比位置开关。
    logits_without_position, debug_without_position = model(comparison_images, use_position=False, return_debug=True)  # 故意移除位置向量执行失败前向。
    logits_with_position, debug_with_position = model(comparison_images, use_position=True, return_debug=True)  # 恢复训练时位置向量执行正确前向。
patch_multiset_one = torch.sort(debug_without_position["flat_patches"][0].sum(dim=1)).values  # 对首图 patch 像素和排序形成无序集合签名。
patch_multiset_two = torch.sort(debug_without_position["flat_patches"][1].sum(dim=1)).values  # 对第二图生成同样无序集合签名。
no_position_difference = float((logits_without_position[0] - logits_without_position[1]).abs().max().item())  # 计算无位置 CLS logits 最大差异。
with_position_difference = float((logits_with_position[0] - logits_with_position[1]).abs().max().item())  # 计算加入位置后的 logits 最大差异。
print(f"错误行为：use_position=False，两个logits={logits_without_position.tolist()}，max_diff={no_position_difference:.8f}")  # 展示位置不同图被视作同一 patch 集合。
print(f"修正行为：use_position=True，两个logits={logits_with_position.tolist()}，max_diff={with_position_difference:.4f}")  # 展示位置向量恢复空间区分。

错误行为：use_position=False，两个logits=[[-5.707174301147461, 5.3843841552734375, 1.6186928749084473], [-5.707174301147461, 5.384384632110596, 1.6186928749084473]]，max_diff=0.00000048
修正行为：use_position=True，两个logits=[[6.2631707191467285, -3.219531774520874, -3.651784658432007], [-2.273003578186035, 7.774305820465088, -4.087771415710449]]，max_diff=10.9938


## 6. 生产边界

十二张规则图会被模型记忆。生产 ViT 需要真实训练/验证/测试切分、RandAugment/Mixup、预训练或蒸馏、分辨率变化时二维位置插值、attention 二次复杂度预算、FlashAttention 或窗口注意力、混合精度稳定性、量化导出，以及按扫描设备监控准确率、校准和 patch 亮度漂移。

In [6]:
vit_diagnostics = {"images": len(images), "patches_per_image": 16, "tokens_with_cls": 17, "baseline_accuracy": baseline_accuracy, "vit_accuracy": vit_accuracy, "initial_loss": history[0]["loss"], "final_loss": history[-1]["loss"], "without_position_logit_difference": no_position_difference, "with_position_logit_difference": with_position_difference}  # 汇总数据、训练、attention 和位置失败指标。
print("生产监控快照：", vit_diagnostics)  # 输出 ViT 分类系统应持续观察的信号。

生产监控快照： {'images': 12, 'patches_per_image': 16, 'tokens_with_cls': 17, 'baseline_accuracy': 0.3333333432674408, 'vit_accuracy': 1.0, 'initial_loss': 1.1775623559951782, 'final_loss': 8.50616124807857e-05, 'without_position_logit_difference': 4.76837158203125e-07, 'with_position_logit_difference': 10.993837356567383}


## 7. 最小回归测试

最后一格只保护图像规模、真实训练、注意力归一化、分类收益和位置编码失败修正。

In [7]:
assert len(images) >= 6 and images.shape == (12, 1, 16, 16)  # 保证包含足够多可显示位置图像。
assert history[-1]["loss"] < history[0]["loss"] and all(row["gradient_norm"] > 0.0 for row in history)  # 保证完整 ViT 真实 backward 学习。
assert final_attention.shape == (12, 4, 17, 17) and torch.allclose(final_attention.sum(dim=-1), torch.ones(12, 4, 17), atol=1.0e-5)  # 保证手写多头注意力 shape 和 softmax 正确。
assert vit_accuracy > baseline_accuracy and vit_accuracy >= 0.90  # 保证位置模型明显优于全局亮度基线。
assert torch.allclose(patch_multiset_one, patch_multiset_two) and no_position_difference < 1.0e-5  # 保证无位置模型对相同 patch 集合无法区分。
assert with_position_difference > 0.1 and final_predictions[0] != final_predictions[4]  # 保证位置向量恢复不同空间类别输出。